# HELENA CPU Quickstart

This quickstart begins with HELENA's standalone leptonic emission workflow. The CPU
and GPU versions use the same scientific calculation; only the execution device differs.

## Before you begin

Follow the [CPU installation instructions](../README.md#installation-and-device-setup), then select a Jupyter kernel from the same environment in which HELENA is installed. Keep this notebook in `quickstart/` with `quickstart_helpers.py`, `notebook_markdown.css`, and the accompanying `.npz` reference files, preserving the release layout.

In a terminal using that environment, run:

```bash
helena cache fetch
helena cache path
helena cache status
```

HELENA v0.1.1 requires matching internal/external BH and gamma-gamma responses. Download them explicitly with `helena cache fetch`, then check `helena cache status`; `helena cache build` is the local alternative. The published v0.1.0 cache is incompatible. For offline/HPC storage, copy the complete validated current cache and select it with `cache_dir=` or `HELENA_CACHE_DIR`. Normal forwards never download or rebuild it. See the [cache instructions](../README.md#response-cache-lifecycle).

## Leptonic emission: Thomson and Klein–Nishina SSC regimes

The benchmark parameters follow [Cerruti et al. (2026)](https://arxiv.org/abs/2411.14218).
The HELENA total curve combines the synchrotron and SSC emission computed below.

Markdown requests Times New Roman from the browser after the presentation setup; code and math retain their own fonts. Keep the shared `quickstart/notebook_markdown.css` in the repository layout. GitHub previews may ignore this styling.


This maintained source targets the v0.1.1 solver. Execution outputs are cleared; rerun in the matching environment. Historical v0.1.0 outputs are not current numerical regression targets.


In [ ]:
import os
os.environ.setdefault("XLA_PYTHON_CLIENT_PREALLOCATE", "false")

import time

try:
    from quickstart import quickstart_helpers as helper
except ModuleNotFoundError:
    import quickstart_helpers as helper

notebook_started = time.perf_counter()
style = helper.setup_quickstart_style()

from helena import configure

DEVICE = "cpu"
runtime = dict(configure(device=DEVICE))

import jax.numpy as jnp
import helena.physics as physics

cases = helper.benchmark_cases()
references = helper.load_benchmark_references()
print(f"Device: {runtime['resolved_device']} (x64={runtime['x64']})")

In [ ]:
def calculate(case):
    energy = case["photon_energy_blob_erg"]
    gamma = case["electron_gamma"]
    electrons = case["electron_density_cm3"]

    synchrotron = physics.electron_synchrotron(
        energy,
        gamma,
        electrons,
        magnetic_field_G=case["magnetic_field_G"],
        radius_cm=case["radius_cm"],
    )
    escaped_synchrotron = physics.electron_synchrotron_self_absorbed_sed(
        energy,
        gamma,
        electrons,
        magnetic_field_G=case["magnetic_field_G"],
        radius_cm=case["radius_cm"],
    )
    absorption = physics.electron_synchrotron_absorption_coefficient(
        energy,
        gamma,
        electrons,
        magnetic_field_G=case["magnetic_field_G"],
    )
    internal_photons = physics.internal_photon_density(
        energy,
        synchrotron,
        jnp.maximum(absorption, 0.0),
        radius_cm=case["radius_cm"],
    )
    ssc = physics.inverse_compton(
        energy,
        gamma,
        electrons,
        energy,
        internal_photons,
        radius_cm=case["radius_cm"],
    )
    frequency, escaped_synchrotron, ssc = helper.observer_components(
        case["frequency_blob_hz"], escaped_synchrotron, ssc, case["doppler_factor"]
    )
    total = escaped_synchrotron + ssc
    return {"frequency_hz": frequency, "total_nu_lnu_erg_s": total}


results, timings = {}, {}
for scenario, case in cases.items():
    _, first_seconds = helper.timed_call(calculate, case)
    result, warm_seconds = helper.timed_call(calculate, case)
    results[scenario] = result
    timings[scenario] = {
        "first_seconds": first_seconds,
        "warm_seconds": warm_seconds,
    }

In [ ]:
main_figure = helper.plot_leptonic_benchmark(results, references)

In [ ]:
helper.show_parameter_table()

In SSC-TH, SSC scattering remains predominantly in the Thomson regime. In SSC-KN,
the higher electron cutoff moves the IC emission into the Klein–Nishina regime,
suppressing the high-energy SSC output. The three codes show consistent peak
ordering and overall SED structure, with implementation-dependent differences
around the high-energy cutoff.

In [ ]:
notebook_total_seconds = time.perf_counter() - notebook_started
helper.show_timing_table(timings, notebook_total_seconds)
print("Times are single synchronized executions, not benchmarks.")
print("Next step: adjust a benchmark parameter and rerun the same Public calls.")

## Electron cooling and transport

Injected electrons undergo synchrotron cooling and escape, producing a steady-state electron distribution and its synchrotron emission.

In [ ]:
import helena.transport

syncool = helper.syncool_inputs(jnp)
syncool_electrons = helena.transport.solve_steady_state(
    syncool["electron_gamma"],
    syncool["injection"],
    syncool["cooling"],
    syncool["escape"],
)

syncool_synchrotron_blob = helena.physics.electron_synchrotron_self_absorbed_sed(
    syncool["photon_energy_blob_erg"],
    syncool["electron_gamma"],
    syncool_electrons,
    magnetic_field_G=syncool["magnetic_field_G"],
    radius_cm=syncool["radius_cm"],
)
syncool_result = helper.syncool_observer_result(
    syncool, syncool_electrons, syncool_synchrotron_blob
)
syncool_figure = helper.plot_syncool_transport(
    syncool_result, syncool["reference"]
)

## Proton–photon interactions

A prescribed pγ-PLPL proton population interacts with a fixed power-law photon target. Proton, pion, and muon cooling are disabled, so the injected pions feed the decay chain to photons, secondary $e^\pm$, and all-flavor neutrinos.

In [ ]:
pgamma_case = helper.pgamma_plpl_case()
pgamma_reference = helper.load_pgamma_reference()

pgamma_pions = physics.photomeson_pion_injection(
    pgamma_case["pion_gamma"],
    pgamma_case["proton_gamma"],
    pgamma_case["proton_density_cm3"],
    pgamma_case["target_photon_energy_erg"],
    pgamma_case["target_photon_density_cm3_erg"],
)
pgamma_pi0_photons = physics.neutral_pion_decay_photon_injection(
    pgamma_case["secondary_energy_erg"],
    pgamma_case["pion_gamma"],
    pgamma_pions.neutral_pion_dgamma_cm3_s,
)

# With no competing pion/muon loss, each injection is its decay-event rate.
pgamma_muons = physics.charged_pion_decay_muon_injection(
    pgamma_case["muon_gamma"],
    pgamma_case["pion_gamma"],
    pgamma_pions.positive_pion_dgamma_cm3_s,
    pgamma_pions.negative_pion_dgamma_cm3_s,
)
pgamma_pion_neutrinos = physics.charged_pion_decay_neutrino_injection(
    pgamma_case["secondary_energy_erg"],
    pgamma_case["pion_gamma"],
    pgamma_pions.positive_pion_dgamma_cm3_s,
    pgamma_pions.negative_pion_dgamma_cm3_s,
)
pgamma_leptons = physics.muon_decay_electron_positron_injection(
    pgamma_case["secondary_energy_dimensionless"],
    pgamma_case["muon_gamma"],
    *pgamma_muons,
)
pgamma_muon_neutrinos = physics.muon_decay_neutrino_injection(
    pgamma_case["secondary_energy_erg"],
    pgamma_case["muon_gamma"],
    *pgamma_muons,
)

pgamma_result = helper.pgamma_secondary_result(
    pgamma_case,
    pgamma_pi0_photons,
    pgamma_leptons,
    pgamma_pion_neutrinos,
    pgamma_muon_neutrinos,
)
pgamma_figure = helper.plot_pgamma_plpl(pgamma_result, pgamma_reference)
helper.show_pgamma_parameter_table()

## Hadronic emission: proton-synchrotron regime

Primary electrons and protons cool and escape on $R/c$ while synchrotron, internal inverse Compton, Bethe–Heitler, photomeson, and secondary-decay emission remain active; internal $\gamma\gamma$ absorption and pair production are disabled for this benchmark.

The comparison is quantity-matched before plotting: photons are escaped broadband observer-frame $\nu L_\nu=E L_E$ in erg s$^{-1}$; neutrinos are the observer-frame sum of all neutrino and antineutrino flavors in the same $\nu L_\nu$ quantity and units. The static AM3 and LeHaMoC curves use only their documented flux-to-luminosity and blob-to-observer conversions—no injection source, transported population, density, or flux is compared as a luminosity.

The retained `enable_ssc` switch now controls internal inverse Compton on the complete internal photon density. True synchrotron-seed SSC remains a nonadditive diagnostic.


In [ ]:
from helena import ModelConfig, OneZoneModel

ps_case = helper.ps_case()
ps_config = ModelConfig(
    magnetic_field_G=ps_case["magnetic_field_G"],
    radius_cm=ps_case["radius_cm"],
    redshift=ps_case["redshift"],
    doppler_factor=ps_case["doppler_factor"],
    luminosity_distance_cm=ps_case["luminosity_distance_cm"],
    electron_kinetic_injection_luminosity_erg_s=(
        ps_case["electron_injection_luminosity_erg_s"]
    ),
    electron_injection_index=ps_case["electron_index"],
    electron_gamma_max=ps_case["electron_gamma_max"],
    proton_kinetic_injection_luminosity_erg_s=(
        ps_case["proton_injection_luminosity_erg_s"]
    ),
    proton_injection_index=ps_case["proton_index"],
    proton_gamma_max=ps_case["proton_gamma_max"],
    enable_ssc=True,
    enable_bethe_heitler=True,
    enable_proton_proton=False,
    cold_proton_density_cm3=0.0,
    enable_pion_synchrotron_photons=True,
    enable_muon_synchrotron_photons=True,
    enable_internal_gamma_gamma=False,
)
ps_model = OneZoneModel(ps_config, cache_dir=ps_case["cache_dir"])
ps_forward = ps_model.forward()

ps_result = helper.observer_multimessenger_result(
    ps_forward, ps_case["luminosity_distance_cm"]
)
ps_reference = helper.load_ps_reference()
ps_figure = helper.plot_ps_benchmark(ps_result, ps_reference)


## Full lepto-hadronic model

Primary electrons and protons cool and escape on $R/c$ while electron/proton synchrotron, internal inverse Compton, Bethe–Heitler, photomeson, secondary decays, pion/muon synchrotron, and internal $\gamma\gamma$ reprocessing remain active; pp interactions and external photon fields are absent.

The comparison is quantity-matched before plotting: photons are escaped broadband observer-frame $\nu L_\nu=E L_E$ in erg s$^{-1}$; neutrinos are the observer-frame sum of all neutrino and antineutrino flavors in the same quantity and units. The static AM3 and LeHaMoC curves use only their documented flux-to-luminosity and blob-to-observer conversions—source injection $Q$, transported population $N$, density, flux, and luminosity are not interchanged.

The retained `enable_ssc` switch now controls internal inverse Compton on the complete internal photon density. True synchrotron-seed SSC remains a nonadditive diagnostic.


In [ ]:
leha_case = helper.leha_case()
leha_config = ModelConfig(
    magnetic_field_G=leha_case["magnetic_field_G"],
    radius_cm=leha_case["radius_cm"],
    redshift=leha_case["redshift"],
    doppler_factor=leha_case["doppler_factor"],
    luminosity_distance_cm=leha_case["luminosity_distance_cm"],
    electron_kinetic_injection_luminosity_erg_s=(
        leha_case["electron_injection_luminosity_erg_s"]
    ),
    electron_injection_index=leha_case["electron_index"],
    electron_gamma_max=leha_case["electron_gamma_max"],
    proton_kinetic_injection_luminosity_erg_s=(
        leha_case["proton_injection_luminosity_erg_s"]
    ),
    proton_injection_index=leha_case["proton_index"],
    proton_gamma_max=leha_case["proton_gamma_max"],
    enable_ssc=True,
    enable_bethe_heitler=True,
    enable_proton_proton=False,
    cold_proton_density_cm3=0.0,
    enable_pion_synchrotron_photons=True,
    enable_muon_synchrotron_photons=True,
    enable_internal_gamma_gamma=True,
)
leha_model = OneZoneModel(leha_config, cache_dir=leha_case["cache_dir"])
leha_forward = leha_model.forward()

leha_result = helper.observer_multimessenger_result(
    leha_forward, leha_case["luminosity_distance_cm"]
)
leha_reference = helper.load_leha_reference()
leha_figure = helper.plot_leha_benchmark(leha_result, leha_reference)


## One-parameter fitting

Eighteen fixed frequencies sample the Full LeHa broadband photon SED from electron synchrotron through the Bethe–Heitler valley, internal inverse-Compton/secondary emission, and the high-energy pion-decay structure. A fixed random seed produces one reproducible 10% Gaussian realization. This is a broadband conditional fit of the electron injection luminosity. All other parameters are fixed.


In [ ]:
from dataclasses import replace

from helena import SEDDataset, fit

fit_mock = helper.broadband_mock_data(
    leha_forward, random_seed=20260824, relative_uncertainty=0.10
)
le_true = leha_config.electron_kinetic_injection_luminosity_erg_s
le_initial = 0.8 * le_true
initial_config = replace(
    leha_config,
    electron_kinetic_injection_luminosity_erg_s=le_initial,
)
initial_model = OneZoneModel(initial_config, cache_dir=leha_case["cache_dir"])

fit_dataset = SEDDataset(
    energy_erg=fit_mock["energy_erg"],
    sed_flux_erg_cm2_s=fit_mock["observed_flux"],
    energy_error_lower_erg=fit_mock["energy_error_erg"],
    energy_error_upper_erg=fit_mock["energy_error_erg"],
    sed_flux_error_lower_erg_cm2_s=fit_mock["flux_uncertainty"],
    sed_flux_error_upper_erg_cm2_s=fit_mock["flux_uncertainty"],
    instrument=fit_mock["instrument"],
    statistic_type=fit_mock["statistic_type"],
    scientific_use=fit_mock["scientific_use"],
)

initial_forward = initial_model.forward()
fit_result = fit(
    initial_model,
    fit_dataset,
    parameters=["log_Le"],
    optimizer="lbfgsb",
)
best_model = OneZoneModel(fit_result.best_config, cache_dir=leha_case["cache_dir"])
best_forward = best_model.forward()

le_best = fit_result.best_parameters["log_Le"]
assert le_initial == 0.8 * le_true
assert fit_result.success and fit_result.free_parameters == ("log_Le",)
assert fit_result.joint_dimension == 1
assert helper.log_ratio_error(le_best, le_true) < 0.10
assert helper.log_ratio_error(le_best, le_true) < helper.log_ratio_error(
    le_initial, le_true
)

helper.show_fit_recovery_table(
    [("L<sub>e</sub> [erg s<sup>−1</sup>]", le_initial, le_true, le_best)],
    first_header="Quantity",
)
print(
    f"success={fit_result.success}; status={fit_result.status}; "
    f"NLL={fit_result.nll:.8g}; iterations={fit_result.iterations}; "
    f"evaluations={fit_result.evaluations}; "
    f"fit runtime={fit_result.runtime_seconds:.2f} s"
)
print(
    f"broadband points={fit_mock['frequency_hz'].size}; "
    f"seed={fit_mock['random_seed']}; "
    f"relative uncertainty={fit_mock['relative_uncertainty']:.0%}"
)
print(
    "This is a broadband conditional fit of the electron injection luminosity. "
    "All other validated parameters are fixed."
)

fit_diagnostics = helper.broadband_fit_diagnostics(
    fit_mock, initial_forward, best_forward
)
assert (abs(fit_diagnostics["best_residual"]) > 1.0e-6).any()
fit_figure = helper.plot_broadband_fit(fit_mock, fit_diagnostics)
fit_figure
